In [1]:
import rasterio
from pathlib import Path

s2_path = Path("/shared-docker/data/makeathon-challenge/sentinel-2/train/18NWG_6_6__s2_l2a/18NWG_6_6__s2_l2a_2020_1.tif")

with rasterio.open(s2_path) as src:
    print("count:", src.count)
    print("descriptions:", src.descriptions)
    print("tags:", src.tags())

count: 12
descriptions: ('5e72bf4c-ece5-4b82-befd-7d8be3cc7af0:B01', '5e72bf4c-ece5-4b82-befd-7d8be3cc7af0:B02', '5e72bf4c-ece5-4b82-befd-7d8be3cc7af0:B03', '5e72bf4c-ece5-4b82-befd-7d8be3cc7af0:B04', '5e72bf4c-ece5-4b82-befd-7d8be3cc7af0:B05', '5e72bf4c-ece5-4b82-befd-7d8be3cc7af0:B06', '5e72bf4c-ece5-4b82-befd-7d8be3cc7af0:B07', '5e72bf4c-ece5-4b82-befd-7d8be3cc7af0:B08', '5e72bf4c-ece5-4b82-befd-7d8be3cc7af0:B8A', '5e72bf4c-ece5-4b82-befd-7d8be3cc7af0:B09', '5e72bf4c-ece5-4b82-befd-7d8be3cc7af0:B11', '5e72bf4c-ece5-4b82-befd-7d8be3cc7af0:B12')
tags: {'AREA_OR_POINT': 'Area'}


In [2]:
from pathlib import Path
import rasterio
import pandas as pd
from collections import Counter, defaultdict

root = Path("/shared-docker/data/makeathon-challenge/sentinel-2")

expected_bands = [
    "B01", "B02", "B03", "B04", "B05", "B06",
    "B07", "B08", "B8A", "B09","B10" ,"B11", "B12"
]

records = []
missing_counter = Counter()
present_counter = Counter()
unexpected_counter = Counter()

s2_files = sorted(root.rglob("*.tif"))

for fp in s2_files:
    with rasterio.open(fp) as src:
        desc = list(src.descriptions or [])
        
        # descriptions look like "...:B01", so keep the suffix after ":"
        bands = []
        for d in desc:
            if d is None:
                continue
            band = d.split(":")[-1]
            bands.append(band)

        band_set = set(bands)
        missing = [b for b in expected_bands if b not in band_set]
        unexpected = [b for b in band_set if b not in expected_bands]

        for b in expected_bands:
            if b in band_set:
                present_counter[b] += 1
            else:
                missing_counter[b] += 1

        for b in unexpected:
            unexpected_counter[b] += 1

        records.append({
            "file": str(fp),
            "split": fp.parts[-4] if len(fp.parts) >= 4 else None,   # train/test
            "tile_dir": fp.parent.name,
            "filename": fp.name,
            "count": src.count,
            "all_expected_present": len(missing) == 0,
            "n_missing": len(missing),
            "missing_bands": ",".join(missing),
            "unexpected_bands": ",".join(unexpected),
            "descriptions": "|".join(bands),
        })

df = pd.DataFrame(records)

print("Total S2 files:", len(df))
print("Files with all expected bands:", int(df["all_expected_present"].sum()))
print("Files missing at least one expected band:", int((~df["all_expected_present"]).sum()))

summary_rows = []
n_files = len(df)

for b in expected_bands:
    summary_rows.append({
        "band": b,
        "files_present": present_counter[b],
        "files_missing": missing_counter[b],
        "missing_pct": 100 * missing_counter[b] / max(n_files, 1),
    })

band_summary = pd.DataFrame(summary_rows).sort_values(["missing_pct", "band"], ascending=[False, True])

print("\nBand availability summary:")
display(band_summary)

print("\nExamples of problematic files:")
display(df.loc[~df["all_expected_present"], ["file", "count", "missing_bands", "unexpected_bands"]].head(20))

Total S2 files: 1495
Files with all expected bands: 0
Files missing at least one expected band: 1495

Band availability summary:


,band,files_present,files_missing,missing_pct
10,B10,0,1495,100.0
0,B01,1495,0,0.0
1,B02,1495,0,0.0
2,B03,1495,0,0.0
3,B04,1495,0,0.0
4,B05,1495,0,0.0
5,B06,1495,0,0.0
6,B07,1495,0,0.0
7,B08,1495,0,0.0
9,B09,1495,0,0.0



Examples of problematic files:


,file,count,missing_bands,unexpected_bands
0,/shared-docker/data/makeathon-challenge/sentin...,12,B10,
1,/shared-docker/data/makeathon-challenge/sentin...,12,B10,
2,/shared-docker/data/makeathon-challenge/sentin...,12,B10,
3,/shared-docker/data/makeathon-challenge/sentin...,12,B10,
4,/shared-docker/data/makeathon-challenge/sentin...,12,B10,
5,/shared-docker/data/makeathon-challenge/sentin...,12,B10,
6,/shared-docker/data/makeathon-challenge/sentin...,12,B10,
7,/shared-docker/data/makeathon-challenge/sentin...,12,B10,
8,/shared-docker/data/makeathon-challenge/sentin...,12,B10,
9,/shared-docker/data/makeathon-challenge/sentin...,12,B10,


In [3]:
tile_summary = (
    df.groupby("tile_dir")
      .agg(
          n_files=("file", "count"),
          complete_files=("all_expected_present", "sum"),
          incomplete_files=("all_expected_present", lambda x: (~x).sum()),
      )
      .reset_index()
)

tile_summary["incomplete_pct"] = 100 * tile_summary["incomplete_files"] / tile_summary["n_files"]
display(tile_summary.sort_values("incomplete_pct", ascending=False))

,tile_dir,n_files,complete_files,incomplete_files,incomplete_pct
0,.ipynb_checkpoints,2,0,2,100.0
1,18NVJ_1_6__s2_l2a,71,0,71,100.0
2,18NWG_6_6__s2_l2a,72,0,72,100.0
3,18NWH_1_4__s2_l2a,72,0,72,100.0
4,18NWJ_8_9__s2_l2a,71,0,71,100.0
5,18NWM_9_4__s2_l2a,72,0,72,100.0
6,18NXH_6_8__s2_l2a,71,0,71,100.0
7,18NXJ_7_6__s2_l2a,72,0,72,100.0
8,18NYH_2_1__s2_l2a,70,0,70,100.0
9,18NYH_9_9__s2_l2a,72,0,72,100.0


In [4]:
There are no B10 channel.

SyntaxError: invalid syntax (2446708386.py, line 1)

In [5]:
import numpy as np
from scipy.ndimage import binary_opening, binary_closing, binary_dilation

S2_BANDS = {
    "B01": 0,
    "B02": 1,
    "B03": 2,
    "B04": 3,
    "B05": 4,
    "B06": 5,
    "B07": 6,
    "B08": 7,
    "B8A": 8,
    "B09": 9,
    "B11": 10,
    "B12": 11,
}

def to_reflectance(x):
    x = x.astype(np.float32)
    if np.nanpercentile(x, 99) > 2:
        x = x / 10000.0
    return x

def rescale(x, lo, hi):
    return np.clip((x - lo) / (hi - lo + 1e-6), 0, 1)

def detect_clouds_s2_v2(s2_cube, cleanup=True):
    b01 = to_reflectance(s2_cube[S2_BANDS["B01"]])
    b02 = to_reflectance(s2_cube[S2_BANDS["B02"]])
    b03 = to_reflectance(s2_cube[S2_BANDS["B03"]])
    b04 = to_reflectance(s2_cube[S2_BANDS["B04"]])
    b08 = to_reflectance(s2_cube[S2_BANDS["B08"]])
    b09 = to_reflectance(s2_cube[S2_BANDS["B09"]])
    b11 = to_reflectance(s2_cube[S2_BANDS["B11"]])
    b12 = to_reflectance(s2_cube[S2_BANDS["B12"]])

    invalid = (
        ~np.isfinite(b02) | ~np.isfinite(b03) | ~np.isfinite(b04) |
        ~np.isfinite(b08) | ~np.isfinite(b11) | ~np.isfinite(b12) |
        (b02 <= 0) | (b03 <= 0) | (b04 <= 0) |
        (b08 <= 0) | (b11 <= 0) | (b12 <= 0)
    )

    ndvi = (b08 - b04) / (b08 + b04 + 1e-6)
    vis = (b02 + b03 + b04) / 3.0
    swir = (b11 + b12) / 2.0
    whiteness = 1.0 - (
        (np.abs(b02 - vis) + np.abs(b03 - vis) + np.abs(b04 - vis)) / (vis + 1e-6)
    )
    whiteness = np.clip(whiteness, 0, 1)

    # Score components
    vis_score = rescale(vis, 0.12, 0.35)
    aerosol_score = rescale(b01, 0.08, 0.25)
    vapor_score = rescale(b09, 0.04, 0.15)
    swir_score = rescale(swir, 0.05, 0.18)
    ndvi_penalty = 1.0 - rescale(ndvi, 0.2, 0.7)

    # Final cloud score
    cloud_score = (
        0.35 * vis_score +
        0.15 * aerosol_score +
        0.10 * vapor_score +
        0.15 * swir_score +
        0.15 * whiteness +
        0.10 * ndvi_penalty
    )

    # Two-stage mask:
    # thick clouds
    thick_cloud = cloud_score > 0.45

    # thin/hazy cloud
    thin_cloud = (
        (cloud_score > 0.22) &
        (vis > 0.08) &
        (ndvi < 0.55)
    )
    cloud_mask = thick_cloud | thin_cloud

    if cleanup:
        cloud_mask = binary_opening(cloud_mask, structure=np.ones((3, 3)))
        cloud_mask = binary_closing(cloud_mask, structure=np.ones((5, 5)))
        cloud_mask = binary_dilation(cloud_mask, structure=np.ones((3, 3)))

    bad_mask = invalid | cloud_mask
    return bad_mask.astype(np.uint8), cloud_mask.astype(np.uint8), cloud_score.astype(np.float32)

In [6]:
import matplotlib.pyplot as plt

bad_mask, cloud_mask, cloud_score = detect_clouds_s2_v2(s2_cube)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(rgb)
axes[0].set_title("S2 RGB")

axes[1].imshow(cloud_score, cmap="viridis")
axes[1].set_title("Cloud score")

axes[2].imshow(cloud_mask, cmap="gray")
axes[2].set_title("Cloud mask v2")

axes[3].imshow(rgb)
axes[3].imshow(cloud_mask, cmap="Reds", alpha=0.3)
axes[3].set_title("Overlay")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

NameError: name 's2_cube' is not defined

In [7]:
from pathlib import Path
import numpy as np
import rasterio

# assumes detect_clouds_s2_v2(s2_cube) already exists

S2_ROOT = Path("/shared-docker/data/makeathon-challenge/sentinel-2")
OUT_ROOT = Path("/shared-docker/cache/s2_cloudmask")

def save_s2_cloudmask_for_file(s2_path: Path, split: str):
    tile_dir = s2_path.parent.name
    out_dir = OUT_ROOT / split / tile_dir
    out_dir.mkdir(parents=True, exist_ok=True)

    out_name = s2_path.stem + "_cloudmask.tif"
    out_path = out_dir / out_name

    with rasterio.open(s2_path) as src:
        s2_cube = src.read()
        _, cloud_mask, cloud_score = detect_clouds_s2_v2(s2_cube)

        profile = src.profile.copy()
        profile.update(
            count=1,
            dtype="uint8",
            nodata=0,
            compress="lzw"
        )

        with rasterio.open(out_path, "w", **profile) as dst:
            dst.write(cloud_mask.astype(np.uint8), 1)

    return out_path

# example on one file
example = Path("/shared-docker/data/makeathon-challenge/sentinel-2/train/18NWG_6_6__s2_l2a/18NWG_6_6__s2_l2a_2020_1.tif")
out = save_s2_cloudmask_for_file(example, split="train")
print("saved:", out)

saved: /shared-docker/cache/s2_cloudmask/train/18NWG_6_6__s2_l2a/18NWG_6_6__s2_l2a_2020_1_cloudmask.tif


In [8]:
from pathlib import Path
import numpy as np
import rasterio

FEATURE_ROOT = Path("/shared-docker/cache/s2_cloudmasked_features")

def build_s2_valid_mask(s2_cube):
    b2 = s2_cube[1]
    b3 = s2_cube[2]
    b4 = s2_cube[3]
    b8 = s2_cube[7]
    b11 = s2_cube[10]
    b12 = s2_cube[11]

    valid = (
        np.isfinite(b2) & np.isfinite(b3) & np.isfinite(b4) &
        np.isfinite(b8) & np.isfinite(b11) & np.isfinite(b12) &
        (b2 > 0) & (b3 > 0) & (b4 > 0) &
        (b8 > 0) & (b11 > 0) & (b12 > 0)
    )
    return valid

def to_reflectance_cube(s2_cube):
    s2_cube = s2_cube.astype(np.float32)
    if np.nanpercentile(s2_cube[3], 99) > 2:
        s2_cube = s2_cube / 10000.0
    return s2_cube

def safe_index(num, den, usable_mask):
    out = np.full(num.shape, np.nan, dtype=np.float32)
    mask = usable_mask & np.isfinite(num) & np.isfinite(den) & (np.abs(den) > 1e-6)
    out[mask] = num[mask] / den[mask]
    return out

def save_s2_features_for_file(s2_path: Path, split: str):
    tile_dir = s2_path.parent.name
    out_dir = FEATURE_ROOT / split / tile_dir
    out_dir.mkdir(parents=True, exist_ok=True)

    out_name = s2_path.stem + "_features.npz"
    out_path = out_dir / out_name

    with rasterio.open(s2_path) as src:
        s2_cube = src.read()
        s2_cube = to_reflectance_cube(s2_cube)

        valid = build_s2_valid_mask(s2_cube)
        _, cloud_mask, cloud_score = detect_clouds_s2_v2(s2_cube)
        usable = valid & (~cloud_mask.astype(bool))

        b4 = s2_cube[3]
        b8 = s2_cube[7]
        b11 = s2_cube[10]
        b12 = s2_cube[11]

        ndvi = safe_index(b8 - b4, b8 + b4 + 1e-6, usable)
        ndmi = safe_index(b8 - b11, b8 + b11 + 1e-6, usable)
        nbr  = safe_index(b8 - b12, b8 + b12 + 1e-6, usable)

        b8_masked = np.where(usable, b8, np.nan).astype(np.float32)
        b11_masked = np.where(usable, b11, np.nan).astype(np.float32)
        b12_masked = np.where(usable, b12, np.nan).astype(np.float32)

        np.savez_compressed(
            out_path,
            ndvi=ndvi.astype(np.float32),
            ndmi=ndmi.astype(np.float32),
            nbr=nbr.astype(np.float32),
            b8=b8_masked,
            b11=b11_masked,
            b12=b12_masked,
            s2_clear=usable.astype(np.uint8),
            cloud_mask=cloud_mask.astype(np.uint8),
            cloud_score=cloud_score.astype(np.float32),
        )

    return out_path

# example
example = Path("/shared-docker/data/makeathon-challenge/sentinel-2/train/18NWG_6_6__s2_l2a/18NWG_6_6__s2_l2a_2020_1.tif")
out = save_s2_features_for_file(example, split="train")
print("saved:", out)

saved: /shared-docker/cache/s2_cloudmasked_features/train/18NWG_6_6__s2_l2a/18NWG_6_6__s2_l2a_2020_1_features.npz


In [9]:
from pathlib import Path

tile_dir = Path("/shared-docker/data/makeathon-challenge/sentinel-2/train/18NWG_6_6__s2_l2a")
files = sorted(tile_dir.glob("*.tif"))

print("n files:", len(files))

for fp in files[:5]:
    out = save_s2_features_for_file(fp, split="train")
    print(out)

n files: 72
/shared-docker/cache/s2_cloudmasked_features/train/18NWG_6_6__s2_l2a/18NWG_6_6__s2_l2a_2020_1_features.npz
/shared-docker/cache/s2_cloudmasked_features/train/18NWG_6_6__s2_l2a/18NWG_6_6__s2_l2a_2020_10_features.npz
/shared-docker/cache/s2_cloudmasked_features/train/18NWG_6_6__s2_l2a/18NWG_6_6__s2_l2a_2020_11_features.npz
/shared-docker/cache/s2_cloudmasked_features/train/18NWG_6_6__s2_l2a/18NWG_6_6__s2_l2a_2020_12_features.npz
/shared-docker/cache/s2_cloudmasked_features/train/18NWG_6_6__s2_l2a/18NWG_6_6__s2_l2a_2020_2_features.npz


In [11]:
from pathlib import Path
from tqdm.auto import tqdm

S2_ROOT = Path("/shared-docker/data/makeathon-challenge/sentinel-2")
FEATURE_ROOT = Path("/shared-docker/cache/s2_cloudmasked_features")

def expected_feature_path(s2_path: Path, split: str):
    tile_dir = s2_path.parent.name
    out_dir = FEATURE_ROOT / split / tile_dir
    return out_dir / f"{s2_path.stem}_features.npz"

def batch_process_s2_split(split="train", overwrite=False):
    s2_files = sorted([
        p for p in (S2_ROOT / split).rglob("*.tif")
        if ".ipynb_checkpoints" not in str(p)
    ])

    total = len(s2_files)
    done = 0
    skipped = 0
    failed = []

    pbar = tqdm(s2_files, desc=f"S2 {split}", unit="file")

    for fp in pbar:
        out_path = expected_feature_path(fp, split)

        if out_path.exists() and not overwrite:
            skipped += 1
            pbar.set_postfix(done=done, skipped=skipped, failed=len(failed), left=total - (done + skipped + len(failed)))
            continue

        try:
            save_s2_features_for_file(fp, split=split)
            done += 1
        except Exception as e:
            failed.append((str(fp), str(e)))

        pbar.set_postfix(done=done, skipped=skipped, failed=len(failed), left=total - (done + skipped + len(failed)))

    print(f"\n{split.upper()} summary")
    print("total    :", total)
    print("processed:", done)
    print("skipped  :", skipped)
    print("failed   :", len(failed))

    if failed[:10]:
        print("\nFirst failures:")
        for f, e in failed[:10]:
            print(f)
            print("  ", e)

    return failed

In [12]:
failed_train = batch_process_s2_split(split="train", overwrite=False)

S2 train:   0%|          | 0/1150 [00:00<?, ?file/s]


TRAIN summary
total    : 1150
processed: 0
skipped  : 1150
failed   : 0


In [13]:
failed_test = batch_process_s2_split(split="test", overwrite=False)

S2 test:   0%|          | 0/343 [00:00<?, ?file/s]


TEST summary
total    : 343
processed: 0
skipped  : 343
failed   : 0


In [14]:
from pathlib import Path
import numpy as np
import rasterio
from tqdm.auto import tqdm

# =========================
# PATHS
# =========================
DATA_ROOT = Path("/shared-docker/data/makeathon-challenge")
S2_ROOT = DATA_ROOT / "sentinel-2"

PROCESSED_ROOT = DATA_ROOT / "processed"
CLOUDMASK_ROOT = PROCESSED_ROOT / "s2_cloudmask"
CLOUDMASK_ROOT.mkdir(parents=True, exist_ok=True)

# =========================
# PACKAGED S2 BAND MAPPING
# B10 absent
# =========================
S2_BAND_TO_INDEX = {
    "B01": 1,
    "B02": 2,
    "B03": 3,
    "B04": 4,
    "B05": 5,
    "B06": 6,
    "B07": 7,
    "B08": 8,
    "B8A": 9,
    "B09": 10,
    "B11": 11,
    "B12": 12,
}

def safe_ratio(num, den):
    out = np.full_like(num, np.nan, dtype=np.float32)
    valid = np.isfinite(num) & np.isfinite(den) & (np.abs(den) > 1e-6)
    out[valid] = (num[valid] / den[valid]).astype(np.float32)
    return out

def get_band(cube, band_name):
    return cube[S2_BAND_TO_INDEX[band_name] - 1]

def read_s2_cube(s2_path: Path):
    with rasterio.open(s2_path) as src:
        cube = src.read().astype(np.float32)   # [C,H,W]
        profile = src.profile.copy()
    return cube, profile

# =========================
# AGGRESSIVE CLOUD DETECTOR
# =========================
def detect_clouds_s2_aggressive(cube):
    b01 = get_band(cube, "B01")
    b02 = get_band(cube, "B02")
    b03 = get_band(cube, "B03")
    b04 = get_band(cube, "B04")
    b08 = get_band(cube, "B08")
    b09 = get_band(cube, "B09")
    b11 = get_band(cube, "B11")
    b12 = get_band(cube, "B12")

    valid = (
        np.isfinite(b01) & np.isfinite(b02) & np.isfinite(b03) &
        np.isfinite(b04) & np.isfinite(b08) & np.isfinite(b09) &
        np.isfinite(b11) & np.isfinite(b12) &
        (b01 > 0) & (b02 > 0) & (b03 > 0) &
        (b04 > 0) & (b08 > 0) & (b09 > 0) &
        (b11 > 0) & (b12 > 0)
    )

    vis_mean = (b02 + b03 + b04) / 3.0
    swir_mean = (b11 + b12) / 2.0
    ndvi = safe_ratio(b08 - b04, b08 + b04)
    ndsi_like = safe_ratio(b03 - b11, b03 + b11)

    cloud_score = np.zeros_like(b02, dtype=np.float32)

    # More aggressive masking
    cloud_score += (b01 > 1200).astype(np.float32) * 0.30
    cloud_score += (vis_mean > 1500).astype(np.float32) * 0.25
    cloud_score += (b09 > 1000).astype(np.float32) * 0.20
    cloud_score += (swir_mean > 900).astype(np.float32) * 0.10
    cloud_score += (ndvi < 0.35).astype(np.float32) * 0.10
    cloud_score += (ndsi_like < 0.70).astype(np.float32) * 0.05

    # Lower threshold => more aggressive masking
    cloud_mask = ((~valid) | (cloud_score >= 0.30)).astype(np.uint8)  # 1 = cloud/bad
    s2_clear = (cloud_mask == 0).astype(np.float32)

    return cloud_mask, s2_clear, cloud_score

# =========================
# SAVE CLOUD MASK ONLY
# =========================
def process_one_s2_file_cloudmask_only(s2_path: Path, split: str, overwrite: bool = False):
    tile_dir = s2_path.parent.name
    stem = s2_path.stem

    out_dir = CLOUDMASK_ROOT / split / tile_dir
    out_dir.mkdir(parents=True, exist_ok=True)

    mask_path = out_dir / f"{stem}_cloudmask.tif"

    if mask_path.exists() and not overwrite:
        return mask_path, "skipped"

    cube, profile = read_s2_cube(s2_path)
    cloud_mask, _, _ = detect_clouds_s2_aggressive(cube)

    profile.update(
        count=1,
        dtype="uint8",
        nodata=255,
        compress="lzw",
    )

    with rasterio.open(mask_path, "w", **profile) as dst:
        dst.write(cloud_mask, 1)

    return mask_path, "processed"

def batch_process_s2_cloudmasks(split="train", overwrite=False):
    s2_dir = S2_ROOT / split
    s2_files = sorted(s2_dir.glob("*__s2_l2a/*.tif"))

    results = {
        "total": len(s2_files),
        "processed": 0,
        "skipped": 0,
        "failed": 0,
        "failures": [],
    }

    for s2_path in tqdm(s2_files, desc=f"S2 cloudmask {split}", unit="file"):
        try:
            _, status = process_one_s2_file_cloudmask_only(
                s2_path, split=split, overwrite=overwrite
            )
            results[status] += 1
        except Exception as e:
            results["failed"] += 1
            results["failures"].append((str(s2_path), str(e)))

    print(f"\n{split.upper()} summary")
    print(f"total    : {results['total']}")
    print(f"processed: {results['processed']}")
    print(f"skipped  : {results['skipped']}")
    print(f"failed   : {results['failed']}")

    if results["failures"]:
        print("\nFirst failures:")
        for path, err in results["failures"][:5]:
            print(path)
            print("  ", err)

    return results

In [15]:
from pathlib import Path

s2_example = Path("/shared-docker/data/makeathon-challenge/sentinel-2/train/18NWG_6_6__s2_l2a/18NWG_6_6__s2_l2a_2020_1.tif")
mask_path, status = process_one_s2_file_cloudmask_only(s2_example, split="train", overwrite=True)

print("status   :", status)
print("mask_path:", mask_path)

status   : processed
mask_path: /shared-docker/data/makeathon-challenge/processed/s2_cloudmask/train/18NWG_6_6__s2_l2a/18NWG_6_6__s2_l2a_2020_1_cloudmask.tif


In [16]:
import rasterio
import numpy as np

with rasterio.open(mask_path) as src:
    m = src.read(1)

print("unique:", np.unique(m))
print("cloud fraction:", float((m == 1).mean()))
print("clear fraction:", float((m == 0).mean()))

unique: [0 1]
cloud fraction: 0.9099943824925
clear fraction: 0.09000561750749997


In [17]:
import rasterio
import numpy as np

with rasterio.open(mask_path) as src:
    m = src.read(1)

print("unique:", np.unique(m))
print("cloud fraction:", float((m == 1).mean()))
print("clear fraction:", float((m == 0).mean()))

unique: [0 1]
cloud fraction: 0.9099943824925
clear fraction: 0.09000561750749997


In [60]:
from pathlib import Path
import numpy as np
import rasterio
from tqdm.auto import tqdm
from scipy.ndimage import binary_opening, binary_closing, binary_dilation

# =========================
# PATHS
# =========================
DATA_ROOT = Path("/shared-docker/data/makeathon-challenge")
S2_ROOT = DATA_ROOT / "sentinel-2"

PROCESSED_ROOT = DATA_ROOT / "processed"
CLOUDMASK_ROOT = PROCESSED_ROOT / "s2_cloudmask"
CLOUDMASK_ROOT.mkdir(parents=True, exist_ok=True)

# =========================
# BAND MAPPING
# =========================
S2_BANDS = {
    "B01": 0,
    "B02": 1,
    "B03": 2,
    "B04": 3,
    "B05": 4,
    "B06": 5,
    "B07": 6,
    "B08": 7,
    "B8A": 8,
    "B09": 9,
    "B11": 10,
    "B12": 11,
}

def to_reflectance(x):
    x = x.astype(np.float32)
    if np.nanpercentile(x, 99) > 2:
        x = x / 10000.0
    return x

def rescale(x, lo, hi):
    return np.clip((x - lo) / (hi - lo + 1e-6), 0, 1)

def read_s2_cube_and_profile(s2_path: Path):
    with rasterio.open(s2_path) as src:
        cube = src.read().astype(np.float32)  # [C,H,W]
        profile = src.profile.copy()
    return cube, profile

def detect_clouds_s2_v2_moderate(s2_cube, cleanup=True):
    b01 = to_reflectance(s2_cube[S2_BANDS["B01"]])
    b02 = to_reflectance(s2_cube[S2_BANDS["B02"]])
    b03 = to_reflectance(s2_cube[S2_BANDS["B03"]])
    b04 = to_reflectance(s2_cube[S2_BANDS["B04"]])
    b08 = to_reflectance(s2_cube[S2_BANDS["B08"]])
    b09 = to_reflectance(s2_cube[S2_BANDS["B09"]])
    b11 = to_reflectance(s2_cube[S2_BANDS["B11"]])
    b12 = to_reflectance(s2_cube[S2_BANDS["B12"]])

    invalid = (
        ~np.isfinite(b02) | ~np.isfinite(b03) | ~np.isfinite(b04) |
        ~np.isfinite(b08) | ~np.isfinite(b11) | ~np.isfinite(b12) |
        (b02 <= 0) | (b03 <= 0) | (b04 <= 0) |
        (b08 <= 0) | (b11 <= 0) | (b12 <= 0)
    )

    ndvi = (b08 - b04) / (b08 + b04 + 1e-6)
    vis = (b02 + b03 + b04) / 3.0
    swir = (b11 + b12) / 2.0
    whiteness = 1.0 - (
        (np.abs(b02 - vis) + np.abs(b03 - vis) + np.abs(b04 - vis)) / (vis + 1e-6)
    )
    whiteness = np.clip(whiteness, 0, 1)

    vis_score = rescale(vis, 0.10, 0.30)
    aerosol_score = rescale(b01, 0.06, 0.20)
    vapor_score = rescale(b09, 0.03, 0.12)
    swir_score = rescale(swir, 0.04, 0.15)
    ndvi_penalty = 1.0 - rescale(ndvi, 0.25, 0.75)

    cloud_score = (
        0.35 * vis_score +
        0.15 * aerosol_score +
        0.10 * vapor_score +
        0.15 * swir_score +
        0.15 * whiteness +
        0.10 * ndvi_penalty
    )

    thick_cloud = cloud_score > 0.42
    thin_cloud = (
        (cloud_score > 0.28) &
        (vis > 0.10) &
        (ndvi < 0.40)
    )

    cloud_mask = thick_cloud | thin_cloud

    if cleanup:
        cloud_mask = binary_opening(cloud_mask, structure=np.ones((3, 3)))
        cloud_mask = binary_closing(cloud_mask, structure=np.ones((5, 5)))
        cloud_mask = binary_dilation(cloud_mask, structure=np.ones((3, 3)))

    bad_mask = invalid | cloud_mask
    return bad_mask.astype(np.uint8), cloud_mask.astype(np.uint8), cloud_score.astype(np.float32)

def process_one_s2_file_cloudmask_only(s2_path: Path, split: str, overwrite: bool = False):
    tile_dir = s2_path.parent.name
    stem = s2_path.stem

    out_dir = CLOUDMASK_ROOT / split / tile_dir
    out_dir.mkdir(parents=True, exist_ok=True)
    mask_path = out_dir / f"{stem}_cloudmask.tif"

    if mask_path.exists() and not overwrite:
        return mask_path, "skipped"

    cube, profile = read_s2_cube_and_profile(s2_path)
    _, cloud_mask, _ = detect_clouds_s2_v2_moderate(cube, cleanup=True)

    profile.update(count=1, dtype="uint8", nodata=255, compress="lzw")

    with rasterio.open(mask_path, "w", **profile) as dst:
        dst.write(cloud_mask, 1)

    return mask_path, "processed"

def batch_process_s2_cloudmasks(split="train", overwrite=False):
    s2_dir = S2_ROOT / split
    s2_files = sorted(s2_dir.glob("*__s2_l2a/*.tif"))

    results = {
        "total": len(s2_files),
        "processed": 0,
        "skipped": 0,
        "failed": 0,
        "failures": [],
    }

    for s2_path in tqdm(s2_files, desc=f"S2 cloudmask {split}", unit="file"):
        try:
            _, status = process_one_s2_file_cloudmask_only(
                s2_path, split=split, overwrite=overwrite
            )
            results[status] += 1
        except Exception as e:
            results["failed"] += 1
            results["failures"].append((str(s2_path), str(e)))

    print(f"\n{split.upper()} summary")
    print(f"total    : {results['total']}")
    print(f"processed: {results['processed']}")
    print(f"skipped  : {results['skipped']}")
    print(f"failed   : {results['failed']}")

    if results["failures"]:
        print("\nFirst failures:")
        for path, err in results["failures"][:5]:
            print(path)
            print("  ", err)

    return results

In [61]:
train_cloud_results = batch_process_s2_cloudmasks("train", overwrite=False)
test_cloud_results = batch_process_s2_cloudmasks("test", overwrite=False)

S2 cloudmask train:   0%|          | 0/1150 [00:00<?, ?file/s]


TRAIN summary
total    : 1150
processed: 1149
skipped  : 1
failed   : 0


S2 cloudmask test:   0%|          | 0/343 [00:00<?, ?file/s]


TEST summary
total    : 343
processed: 343
skipped  : 0
failed   : 0
